<a href="https://colab.research.google.com/github/Coding-4A11j/-Analyzing-Restaurant-Trends-Insights-into-Cuisines-Ratings-and-Service-O/blob/main/Multi_Object_Tracking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.4 MB/s eta 0:00:00


In [ ]:
import cv2
import os
from ultralytics import YOLO
from google.colab import files

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
uploaded = files.upload()

video_path = list(uploaded.keys())[0]
print("Uploaded file:", /content/videoplayback (1).mp4)

Saving videoplayback (1).mp4 to videoplayback (1).mp4
Uploaded file: videoplayback (1).mp4


In [ ]:
model = YOLO("yolov8n.pt")  # stable & fast

In [ ]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise Exception("Error: Cannot open video")

# Get properties safely
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 640)
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 480)
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Fix FPS issue
if fps == 0:
    fps = 30

output_path = "output.mp4"

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

print(f"Resolution: {width}x{height}, FPS: {fps}")

Resolution: 640x360, FPS: 25


In [14]:
track_history = {}

In [17]:
prev_positions = {}

Now, let's update the main tracking loop to include speed calculation and display.

In [18]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True, classes=[0], verbose=False)

    if results and results[0].boxes is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        ids = results[0].boxes.id

        if ids is not None:
            ids = ids.cpu().numpy()

            for box, track_id in zip(boxes, ids):
                x1, y1, x2, y2 = map(int, box)
                cx = int((x1 + x2) / 2)
                cy = int((y1 + y2) / 2)

                track_id = int(track_id)

                # Speed calculation
                speed = 0
                if track_id in prev_positions:
                    px, py = prev_positions[track_id]
                    speed = int(((cx - px)**2 + (cy - py)**2)**0.5)

                prev_positions[track_id] = (cx, cy)

                # Save trajectory
                if track_id not in track_history:
                    track_history[track_id] = []

                track_history[track_id].append((cx, cy))

                # Keep last 30 points
                if len(track_history[track_id]) > 30:
                    track_history[track_id].pop(0)

                # Draw box
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

                # Draw ID
                cv2.putText(frame, f"ID {track_id}",
                            (x1, y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.7, (0,255,0), 2)

                # Draw speed
                cv2.putText(frame, f"S:{speed}",
                            (x1, y2+20),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.5, (0,255,255), 2)

                # Draw trajectory
                points = track_history[track_id]
                for i in range(1, len(points)):
                    cv2.line(frame, points[i-1], points[i], (0,0,255), 2)

    # Add count display
    count = len(ids) if ids is not None else 0
    cv2.putText(frame, f"Count: {count}",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1, (255,0,0), 2)

    out.write(frame)

In [19]:
files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>